# NIFTY 50 ATM Option Pricing & Greeks Sensitivity Analysis
**Portfolio Project:** `nifty-risk-lab` | **Author:** Aditya Kansal
**Module:** `src/options.py` (European Black-Scholes-Merton Engine from Scratch)

---

### Objective & Setup
This notebook investigates how an **At-The-Money (ATM) NIFTY 50 index option's theoretical price and Greeks** ($\Delta, \Gamma, \mathcal{V}, \Theta, \rho$) respond to variations in:
1. **Annualized Volatility ($\sigma$)**: from low-vol regimes ($10\%$) to high-vol regimes ($40\%$).
2. **Time to Expiration ($T$)**: from ultra-short weekly contracts ($7$ days) to quarterly expiries ($90$ days).

All baseline inputs are taken **strictly from real, cleaned market data** as of the latest completed session:
- **Underlying Index Spot ($S_0$)**: INR 25227.95 (NIFTY 50 close on 2026-11-18)
- **ATM Strike ($K$)**: INR 25250 (nearest standard NSE 50-strike)
- **Market Implied Volatility Benchmark**: 15.57% (India VIX close on 2026-11-18)
- **Risk-Free Rate ($r$)**: 6.50% (proxy for 91-day Government of India Treasury Bills / RBI Repo Rate)
- **Dividends ($q$)**: 0 (European baseline standard)


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure src modules are discoverable
sys.path.insert(0, os.path.abspath(".."))
from src.options import (
    all_greeks,
    black_scholes_price,
    compute_option_surface,
    delta,
    gamma,
    theta,
    vega,
)

# Load aligned clean daily market data
df = pd.read_parquet("../data/clean/market_daily.parquet")
last_row = df.iloc[-1]
latest_date = str(last_row["Date"])[:10]
spot = float(last_row["nifty50_close"])
vix = float(last_row["india_vix_close"])
strike = round(spot / 50.0) * 50.0
rate = 0.065

print("=" * 60)
print("MARKET PARAMETERS (Source: data/clean/market_daily.parquet)")
print("=" * 60)
print(f"Latest Market Date      : {latest_date}")
print(f"NIFTY 50 Spot Price (S) : INR {spot:.2f}")
print(f"ATM Strike (K)          : INR {strike:.0f}")
print(f"India VIX Benchmark     : {vix:.2f}% (σ = {vix / 100:.4f})")
print(f"Risk-free Rate (r)      : {rate * 100:.2f}%")
print("=" * 60)


In [ ]:
# Compute baseline pricing and Greeks across standard NSE expiry tenors
tenors_days = [7, 14, 21, 30, 45, 60, 90]
rows = []

vix_sigma = vix / 100.0

for d in tenors_days:
    t_years = d / 365.0
    c_greeks = all_greeks(spot, strike, rate, vix_sigma, t_years, "call")
    p_greeks = all_greeks(spot, strike, rate, vix_sigma, t_years, "put")

    rows.append({
        "Expiry (Days)": d,
        "Call Price (₹)": round(c_greeks["price"], 2),
        "Put Price (₹)": round(p_greeks["price"], 2),
        "Call Delta": round(c_greeks["delta"], 4),
        "Put Delta": round(p_greeks["delta"], 4),
        "Gamma": round(c_greeks["gamma"], 6),
        "Vega (per 1% σ)": round(c_greeks["vega"] / 100.0, 2),
        "Call Theta (₹/day)": round(c_greeks["theta"] / 365.0, 2),
        "Put Theta (₹/day)": round(p_greeks["theta"] / 365.0, 2),
    })

baseline_df = pd.DataFrame(rows)
baseline_df.set_index("Expiry (Days)")


---
## 1. Sensitivity of ATM Option Prices to Volatility & Expiry
How do option premiums respond when volatility shifts from tranquil regimes ($\sigma = 10\%$) to acute crisis levels ($\sigma = 40\%$)?
- **Linear Scaling with Volatility**: For ATM options, the Black-Scholes price is approximately linear in volatility ($C_{\text{ATM}} \approx 0.4 \cdot S \cdot \sigma \sqrt{T}$).
- **Term Structure Scaling**: Longer-dated options possess substantially higher Vega, meaning their absolute rupee price is far more sensitive to implied volatility repricing.


In [ ]:
# Plot ATM Call and Put Prices vs Volatility across tenors
sigmas = np.linspace(0.10, 0.40, 50)
selected_tenors = [7, 14, 30, 60, 90]

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for d in selected_tenors:
    t = d / 365.0
    call_prices = [black_scholes_price(spot, strike, rate, s, t, "call") for s in sigmas]
    put_prices = [black_scholes_price(spot, strike, rate, s, t, "put") for s in sigmas]

    axes[0].plot(sigmas * 100, call_prices, label=f"{d} Days", lw=2)
    axes[1].plot(sigmas * 100, put_prices, label=f"{d} Days", lw=2)

axes[0].axvline(vix, color="red", linestyle="--", alpha=0.7, label=f"Current VIX ({vix:.1f}%)")
axes[1].axvline(vix, color="red", linestyle="--", alpha=0.7, label=f"Current VIX ({vix:.1f}%)")

axes[0].set_title("ATM Call Price vs Annualized Volatility", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Annualized Volatility σ (%)")
axes[0].set_ylabel("Call Premium (INR)")
axes[0].legend()

axes[1].set_title("ATM Put Price vs Annualized Volatility", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Annualized Volatility σ (%)")
axes[1].set_ylabel("Put Premium (INR)")
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 2. Dynamic Greek Behavior vs. Time to Expiration ($T$)
The Greeks behave distinctly as maturity approaches zero ($T \to 0$):
1. **Gamma ($\Gamma$) Explodes**: As expiry approaches, an ATM option has maximum uncertainty regarding whether it finishes in or out of the money. This creates massive "pin risk" and sharp Delta adjustments.
2. **Theta ($\Theta$) Accelerates ("Decay Cliff")**: Option sellers benefit from accelerating time decay in the final 14 days, while option buyers suffer steep decay.
3. **Vega ($\mathcal{V}$) Evaporates**: As $T \to 0$, there is less time for volatility to alter the outcome, causing Vega to collapse to zero.


In [ ]:
# Greeks vs Time to Expiry (1 to 90 days)
days = np.arange(1, 91)
t_arr = days / 365.0

deltas_call = [delta(spot, strike, rate, vix_sigma, t, "call") for t in t_arr]
gammas = [gamma(spot, strike, rate, vix_sigma, t) for t in t_arr]
vegas_per_pct = [vega(spot, strike, rate, vix_sigma, t) / 100.0 for t in t_arr]
thetas_daily = [theta(spot, strike, rate, vix_sigma, t, "call") / 365.0 for t in t_arr]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Delta
axes[0, 0].plot(days, deltas_call, color="#1f77b4", lw=2)
axes[0, 0].axhline(0.5, color="gray", linestyle=":", alpha=0.6)
axes[0, 0].set_title("ATM Call Delta vs Days to Expiry (Drift Effect)", fontweight="bold")
axes[0, 0].set_xlabel("Days to Expiry")
axes[0, 0].set_ylabel("Delta (Δ)")

# Gamma
axes[0, 1].plot(days, gammas, color="#ff7f0e", lw=2)
axes[0, 1].set_title("ATM Gamma vs Days to Expiry (Expiry Pin Risk)", fontweight="bold")
axes[0, 1].set_xlabel("Days to Expiry")
axes[0, 1].set_ylabel("Gamma (Γ)")

# Vega
axes[1, 0].plot(days, vegas_per_pct, color="#2ca02c", lw=2)
axes[1, 0].set_title("ATM Vega vs Days to Expiry (Proportional to √T)", fontweight="bold")
axes[1, 0].set_xlabel("Days to Expiry")
axes[1, 0].set_ylabel("Vega (INR per 1% vol change)")

# Theta
axes[1, 1].plot(days, thetas_daily, color="#9467bd", lw=2)
axes[1, 1].set_title("ATM Theta Decay vs Days to Expiry (Accelerating Cliff)", fontweight="bold")
axes[1, 1].set_xlabel("Days to Expiry")
axes[1, 1].set_ylabel("Daily Theta Decay (INR/day)")

plt.tight_layout()
plt.show()


---
## 3. Comprehensive 2D Greeks & Price Surfaces
The heatmaps below illustrate the full 2D parameter space across both volatility $\sigma \in [10\%, 40\%]$ and expiry $T \in [7, 90]$ days:
- **Price**: Maximized at high volatility and long tenors.
- **Gamma**: Highly concentrated in the bottom-left corner (short tenors, low/moderate vol).
- **Vega**: Concentrated in the upper-right corner (long tenors, high vol).
- **Theta**: Most punishing in the bottom-left quadrant (near-term contracts at elevated vol).


In [ ]:
# Generate 2D Option Surface across (sigma, tenor) grid
sig_grid = np.linspace(0.10, 0.40, 31)
tenor_grid_days = np.array([7, 14, 21, 30, 45, 60, 90])
tenor_grid_years = tenor_grid_days / 365.0

surface_grid = compute_option_surface(spot, strike, rate, sig_grid, tenor_grid_years, "call")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price
sns.heatmap(
    surface_grid["price"],
    xticklabels=tenor_grid_days,
    yticklabels=[f"{s*100:.0f}%" for s in sig_grid],
    cmap="Blues",
    ax=axes[0, 0],
    cbar_kws={"label": "Call Price (INR)"}
)
axes[0, 0].set_title("ATM Call Price (INR)", fontweight="bold")
axes[0, 0].set_xlabel("Days to Expiry")
axes[0, 0].set_ylabel("Volatility σ")

# Gamma
sns.heatmap(
    surface_grid["gamma"],
    xticklabels=tenor_grid_days,
    yticklabels=[f"{s*100:.0f}%" for s in sig_grid],
    cmap="Oranges",
    ax=axes[0, 1],
    cbar_kws={"label": "Gamma"}
)
axes[0, 1].set_title("Gamma (Curvature)", fontweight="bold")
axes[0, 1].set_xlabel("Days to Expiry")
axes[0, 1].set_ylabel("Volatility σ")

# Vega
sns.heatmap(
    surface_grid["vega"],
    xticklabels=tenor_grid_days,
    yticklabels=[f"{s*100:.0f}%" for s in sig_grid],
    cmap="Greens",
    ax=axes[1, 0],
    cbar_kws={"label": "Vega"}
)
axes[1, 0].set_title("Vega (Vol Sensitivity)", fontweight="bold")
axes[1, 0].set_xlabel("Days to Expiry")
axes[1, 0].set_ylabel("Volatility σ")

# Daily Theta
sns.heatmap(
    surface_grid["theta"] / 365.0,
    xticklabels=tenor_grid_days,
    yticklabels=[f"{s*100:.0f}%" for s in sig_grid],
    cmap="Purples_r",
    ax=axes[1, 1],
    cbar_kws={"label": "Daily Decay (INR/day)"}
)
axes[1, 1].set_title("Daily Theta Decay (INR/day)", fontweight="bold")
axes[1, 1].set_xlabel("Days to Expiry")
axes[1, 1].set_ylabel("Volatility σ")

title_line1 = (
    f"NIFTY 50 Option Surface: Spot INR {spot:.2f}, "
    f"ATM Strike INR {strike:.0f}, r = {rate*100:.1f}%"
)
title_line2 = f"Baseline Benchmark: India VIX = {vix:.2f}%"

plt.suptitle(
    title_line1 + "\n" + title_line2,
    fontsize=13,
    fontweight="bold",
    y=1.02
)
plt.tight_layout()
plt.show()


---
## 4. Key Takeaways & Limitations for Index Options

### Quantitative Takeaways:
1. **Delta Drift**: Because the risk-free rate $r = 6.5\%$ is positive, the ATM call delta is slightly above $0.50$ (typically $\sim 0.52 - 0.55$), and the ATM put delta is between $-0.45$ and $-0.48$. This reflects the positive forward drift $F = S e^{rT}$.
2. **Weekly Option Risk Profile**: Weekly contracts ($T \le 7$ days) exhibit extremely low Vega but explosive Gamma. Risk management for weekly options is predominantly a Gamma/pin-risk exercise rather than a volatility trade.
3. **Monthly / Quarterly Option Risk Profile**: Long-dated contracts ($T \ge 60$ days) are dominant Vega vehicles where even modest 2% shifts in implied volatility overpower daily time decay.

### Limitations of Black-Scholes for NIFTY 50:
- **Implied Volatility Smile/Skew**: BSM assumes constant $\sigma$. In real NSE markets, OTM puts trade at substantial IV premia over ATM calls due to downside crash hedging.
- **Fat Tails & Gap Risk**: Geometric Brownian Motion does not account for overnight gapping or jump discontinuities common in macro-driven index markets.
- **Index Dividends**: Real Nifty constituents pay seasonal dividends which lower forward prices relative to $S e^{rT}$.
*See `docs/METHODS.md` (Section 9) for mathematical derivations and in-depth analysis.*
